# Which abstracts fed the Material Properties knowledge graph?

This notebook extracts the per-edge source DOI stored on every relationship in the Material Properties knowledge graph (`MatProp_62999.graphml`), and cross-references the unique set of DOIs against the literature metadata in `paper_metadata/Properties abstracts/` to determine exactly which abstracts contributed extracted relationships.

It reproduces, as a runnable/verifiable artifact, the finding that the KG traces to exactly 59,154 unique abstracts — not the 62,999 implied by its filename, nor the 63,222 claimed in `paper_metadata/README.md`.

Results are written to `paper_metadata/kg_source_doi_report/matprop_used_dois_part{1,2}.json` — every one of the 59,154 matched DOIs with title, journal, year, and abstract text, split into two parts (~50MB each) to stay under GitHub's 100MB per-file limit.

### Imports and paths

In [26]:
import html
import json
import re
from pathlib import Path

import openpyxl
import xlrd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
KG_DIR = REPO_ROOT / "data" / "MARS_Data" / "KGs"
PAPER_METADATA_DIR = REPO_ROOT / "paper_metadata"
OUT_DIR = PAPER_METADATA_DIR / "kg_source_doi_report"

EDGE_ID_RE = re.compile(r'<data key="d6">([^<]*)</data>')
EDGE_NAN_DOI_RE = re.compile(r'<data key="d7">nan</data>')

### GraphML edge-id extraction

Each extracted relationship (edge) in the KG carries the DOI of the paper it was extracted from, via GraphML key `d6`. The graph additionally carries a `d7` placeholder (`nan`) on edges where no DOI was available, letting us count those explicitly.

In [27]:
def extract_edge_ids(graphml_path: Path) -> tuple[list[str], int]:
    """Returns (list of raw edge-level 'DOI' attribute values, count of edges with no id at all)."""
    text = graphml_path.read_text(encoding="utf-8", errors="replace")
    ids = EDGE_ID_RE.findall(text)
    no_id_count = len(EDGE_NAN_DOI_RE.findall(text))
    return ids, no_id_count

### Reference metadata loader

Loads DOI, title, journal, year, and abstract out of `paper_metadata/Properties abstracts/*.xls` to cross-reference against. These files are a mix of real legacy OLE2 `.xls` and mislabeled modern `.xlsx` files (same extension, different format) — `read_xls_any_metadata` tries openpyxl first (via a file object, to dodge its extension-based rejection) and falls back to `xlrd` for the genuine legacy files.

In [28]:
def read_xls_any_metadata(path: Path, columns: list[str]) -> dict[str, tuple]:
    """Reads a .xls file that may actually be a real legacy OLE2 file (xlrd) or a
    mislabeled modern .xlsx (openpyxl, opened as a file object to dodge the extension
    gate), and returns {normalized DOI: tuple(values for `columns`)}. `columns` must
    include "DOI"."""

    def _rows_to_dict(rows_iter, header):
        idxs = [header.index(c) if c in header else None for c in columns]
        doi_pos = columns.index("DOI")
        out = {}
        for row in rows_iter:
            vals = [row[i] if i is not None else None for i in idxs]
            doi = vals[doi_pos]
            if doi:
                out[str(doi).strip().lower()] = tuple(vals)
        return out

    try:
        wb = openpyxl.load_workbook(open(path, "rb"), read_only=True, data_only=True)
        ws = wb[wb.sheetnames[0]]
        rows = list(ws.iter_rows(values_only=True))
        if not rows:
            return {}
        return _rows_to_dict(rows[1:], list(rows[0]))
    except Exception:
        pass

    book = xlrd.open_workbook(str(path))
    sheet = book.sheet_by_index(0)
    header = [sheet.cell_value(0, c) for c in range(sheet.ncols)]
    rows = ([sheet.cell_value(r, c) for c in range(sheet.ncols)] for r in range(1, sheet.nrows))
    return _rows_to_dict(rows, header)


def load_matprop_reference_metadata() -> dict[str, tuple]:
    """Returns {normalized DOI: (DOI, Article Title, Source Title, Publication Year, Abstract)}
    across all paper_metadata/Properties abstracts/*.xls files."""
    abstracts_dir = PAPER_METADATA_DIR / "Properties abstracts"
    columns = ["DOI", "Article Title", "Source Title", "Publication Year", "Abstract"]
    metadata = {}
    for fp in sorted(abstracts_dir.glob("*.xls")):
        metadata.update(read_xls_any_metadata(fp, columns))
    return metadata

### Matching logic

The KG's edge id is a genuine DOI. The only mismatch source is HTML-entity escaping (`&lt;`/`&gt;` in old CAS/Wiley-style DOIs like `10.1002/1097-4636(2000)53:6<806::aid-jbm23>3.0.co;2-p`), fixed by `html.unescape`.

In [29]:
def analyze_matprop(ids: list[str], reference_dois: set[str]) -> dict:
    unique_raw = set(ids)
    unique_unescaped = {html.unescape(i).strip().lower() for i in unique_raw}
    matched = unique_unescaped & reference_dois
    unmatched = sorted(unique_unescaped - reference_dois)
    return {"unique_ids": unique_unescaped, "matched": matched, "unmatched": unmatched}

## Material Properties KG (`MatProp_62999.graphml`)

In [30]:
mat_ids, mat_no_id = extract_edge_ids(KG_DIR / "MatProp_62999.graphml")
print(f"{len(mat_ids)} edges with id, {mat_no_id} edges with NaN id")

mat_metadata = load_matprop_reference_metadata()
mat_ref = set(mat_metadata.keys())
print(f"{len(mat_ref)} unique reference DOIs in paper_metadata/Properties abstracts/")

mat_result = analyze_matprop(mat_ids, mat_ref)
print(f"{len(mat_result['unique_ids'])} unique source DOIs in the KG")
print(f"{len(mat_result['matched'])} matched ({len(mat_result['matched']) / len(mat_result['unique_ids']):.1%})")

732572 edges with id, 41437 edges with NaN id
160494 unique reference DOIs in paper_metadata/Properties abstracts/
59154 unique source DOIs in the KG
59154 matched (100.0%)


### Write the JSON report

In [31]:
OUT_DIR.mkdir(exist_ok=True)

total_mat_edges = len(mat_ids) + mat_no_id
mat_summary_row = [
    "Material Properties (MatProp_62999.graphml)",
    total_mat_edges, len(mat_ids), mat_no_id, len(mat_result["unique_ids"]),
    len(mat_result["matched"]),
    f"{len(mat_result['matched']) / len(mat_result['unique_ids']):.1%}",
]


def _record_for(d):
    _, title, journal, year, abstract = mat_metadata.get(d, (None, None, None, None, None))
    if isinstance(year, float) and year.is_integer():
        year = int(year)
    return {"doi": d, "title": title, "journal": journal, "year": year, "abstract": abstract}


mat_records = [_record_for(d) for d in sorted(mat_result["matched"])]

# Abstract text makes the full file ~100MB, right at GitHub's per-file push limit,
# so it's split into two roughly equal parts (well under the limit) rather than one file.
midpoint = len(mat_records) // 2
parts = [mat_records[:midpoint], mat_records[midpoint:]]
for i, part in enumerate(parts, start=1):
    with open(OUT_DIR / f"matprop_used_dois_part{i}.json", "w", encoding="utf-8") as f:
        json.dump(part, f, indent=1, ensure_ascii=False)

print(f"Wrote matprop_used_dois_part{{1,2}}.json to {OUT_DIR}")

Wrote matprop_used_dois_part{1,2}.json to /home/palsubhadeep/Github/MARS_backup_20260707_204426/paper_metadata/kg_source_doi_report


### Summary

In [32]:
print(mat_summary_row)

['Material Properties (MatProp_62999.graphml)', 774009, 732572, 41437, 59154, 59154, '100.0%']
